# NEC CartPole Demo

This notebook is a friendly debugging surface for the repo-native Neural Episodic Control workflow. The implementation lives in `model/nec_workflow.py`; this notebook only configures runs, calls workflow functions, and visualizes the artifacts they write.

## Setup

Run this cell first. It imports the maintained NEC workflow functions and makes sure the repository root is importable from the notebook.

In [1]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.nec_workflow import (
    evaluate_nec,
    load_nec_checkpoint,
    make_nec_config,
    train_nec,
)

ROOT

PosixPath('/Users/ElliotMayo/Documents/GitHub/NN-kNN')

## Choose Run Settings

`smoke` is best for a quick end-to-end check. `debug` is the shorter 25k-step development run. `fast` is the DQN-like 150k-step tuned NEC baseline for comparison.

In [2]:
PROFILE = "fast"  # use "debug" for 25k steps or "fast" for the 150k-step tuned baseline
SEED = 0
DEVICE = None  # use None for repo default, or set "cpu" / "cuda"

cfg = make_nec_config(PROFILE, seed=SEED)
cfg

NECConfig(profile='fast', seed=0, total_timesteps=150000, learning_rate=0.001, replay_size=20000, dictionary_size=10000, gamma=0.99, n_step=100, k_neighbors=5, kernel_delta=0.001, batch_size=128, start_e=1.0, end_e=0.05, exploration_fraction=0.5, learning_starts=1000, train_frequency=1, eval_frequency=0, eval_episode_frequency=100, eval_episodes=20, eval_seed=10000, success_threshold=475.0, embedding_dim=32, hidden_sizes=(128, 128), max_grad_norm=10.0, source_reference='Neural Episodic Control paper; EndingCredits/Neural-Episodic-Control as reference only')

## Train NEC

This trains through the configured fixed step budget, evaluates periodically, and saves the best evaluated checkpoint plus the final/end-of-budget evaluation.

In [3]:
state = train_nec("cartpole", cfg, device=DEVICE, progress=True)
summary = state["summary"]
summary

[nec] step=18 episode=1 return=18.0 length=18 epsilon=1.000 loss=None entries=18
[nec] step=32 episode=2 return=14.0 length=14 epsilon=1.000 loss=None entries=32
[nec] step=44 episode=3 return=12.0 length=12 epsilon=0.999 loss=None entries=44
[nec] step=62 episode=4 return=18.0 length=18 epsilon=0.999 loss=None entries=62
[nec] step=85 episode=5 return=23.0 length=23 epsilon=0.999 loss=None entries=85
[nec] step=277 episode=10 return=17.0 length=17 epsilon=0.997 loss=None entries=277
[nec] step=468 episode=20 return=13.0 length=13 epsilon=0.994 loss=None entries=468
[nec] step=710 episode=30 return=13.0 length=13 epsilon=0.991 loss=None entries=710
[nec] step=893 episode=40 return=13.0 length=13 epsilon=0.989 loss=None entries=893
[nec] step=1133 episode=50 return=14.0 length=14 epsilon=0.986 loss=4.200493335723877 entries=1133
[nec] step=1390 episode=60 return=42.0 length=42 epsilon=0.982 loss=2.9671554565429688 entries=1390
[nec] step=1602 episode=70 return=10.0 length=10 epsilon=0.9

KeyboardInterrupt: 

## Inspect Run Artifacts

Each run writes the same artifact set as the DQN workflow: config, training rows, loss rows, eval rows, episode-level eval files, summary, manifest, and checkpoint.

In [ ]:
run_dir = Path(state["run_dir"])
print(run_dir)
for path in sorted(run_dir.iterdir()):
    print(path.name)

with (run_dir / "summary.json").open(encoding="utf-8") as handle:
    saved_summary = json.load(handle)

{
    "selected_step": saved_summary["selected_step"],
    "selected_source": saved_summary["selected_source"],
    "final_eval": saved_summary["final_eval"],
    "last_eval": saved_summary["last_eval"],
    "training_efficiency": saved_summary["training_efficiency"],
}

## Plot Training Returns

Episode returns show what happened during exploration-heavy training. The selected checkpoint comes from periodic greedy evaluations, not necessarily the final training episode.

In [ ]:
training_rows = []
with (run_dir / "training_metrics.csv").open(newline="", encoding="utf-8") as handle:
    training_rows = list(csv.DictReader(handle))

steps = [int(row["global_step"]) for row in training_rows]
returns = [float(row["episode_return"]) for row in training_rows]

plt.figure(figsize=(9, 4))
plt.plot(steps, returns, linewidth=1.25)
plt.xlabel("Global step")
plt.ylabel("Episode return")
plt.title("NEC CartPole training returns")
plt.grid(alpha=0.3)
plt.show()

## Plot Periodic Evaluations

Periodic evaluations use the greedy NEC policy and are the values used for best-checkpoint selection.

In [ ]:
eval_rows = []
with (run_dir / "eval_metrics.csv").open(newline="", encoding="utf-8") as handle:
    eval_rows = list(csv.DictReader(handle))

if eval_rows:
    eval_steps = [int(row["global_step"]) for row in eval_rows]
    eval_returns = [float(row["mean_return"]) for row in eval_rows]

    plt.figure(figsize=(9, 4))
    plt.plot(eval_steps, eval_returns, marker="o")
    plt.axhline(cfg.success_threshold or 0, linestyle="--", color="tab:green", alpha=0.5)
    plt.xlabel("Global step")
    plt.ylabel("Mean eval return")
    plt.title("NEC CartPole periodic evaluations")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No periodic eval rows were written for this profile/run length.")

## Reload Best Checkpoint And Evaluate

This reloads the selected checkpoint from disk and evaluates it again with the configured evaluation seed.

In [ ]:
loaded = load_nec_checkpoint(state["checkpoint_path"], device=DEVICE)
metrics = evaluate_nec(
    loaded["task"]["name"],
    loaded["model"],
    loaded["dnd"],
    loaded["config"],
    episodes=cfg.eval_episodes,
    seed=cfg.eval_seed,
    device=loaded["device"],
)
{k: v for k, v in metrics.items() if k != "episode_metrics"}